In [ ]:
import pandas as pd
import numpy as np
import requests

In [118]:
def fetch_creditcard(X_y_split: bool = False
                     ) -> pd.DataFrame | tuple[pd.DataFrame, pd.Series]:
    response = requests.get("http://pipeline:8000/dataset/creditcard-churn", params={"X_y_split": X_y_split})
    payload = response.json()
    
    if X_y_split:
        X = pd.DataFrame(payload["x"], index=payload["index"])
        y = pd.Series(payload["y"], index=payload["index"])
        return X, y
    
    df = pd.DataFrame(payload["data"], index=payload["index"])
    
    return df

In [119]:
df = fetch_creditcard(X_y_split=False)
df

,creditcard_churn_id,churn,age,gender,dependents,education_id,marital_id,income_id,card_type_id,relationship_months,...,inactive_months,contact_count,credit_limit,revolving_balance,available_credit,amount_change,count_change,transaction_amount,transaction_count,utilization_ratio
0,1,0,62,1,0,2,4,3,4,6,...,4,4,19151.38,2176.98,16970.26,0.8598,0.5036,8009.08,71,0.5956
1,2,0,25,1,1,7,4,4,1,32,...,2,2,16365.94,2813.62,21242.71,1.4505,0.8159,6325.20,72,0.2807
2,3,1,51,0,0,3,4,6,2,42,...,3,1,22347.52,3811.08,8234.43,1.4503,0.7663,4345.77,64,0.5810
3,4,1,53,1,0,1,3,2,2,16,...,1,4,16297.25,1102.93,18681.44,1.0932,1.4136,6808.25,28,0.6436
4,5,0,28,0,2,5,1,5,4,6,...,5,2,19023.06,2676.39,9840.49,0.7028,0.7881,7155.46,36,0.9812
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10222,10250,0,50,1,2,4,1,2,1,40,...,2,3,4003.00,1851.00,2152.00,0.7030,0.8570,15476.00,117,0.4620
10223,10251,1,41,1,2,7,3,2,1,25,...,2,3,4277.00,2186.00,2091.00,0.8040,0.6830,8764.00,69,0.5110
10224,10252,1,44,0,1,2,2,1,1,36,...,3,4,5409.00,0.00,5409.00,0.8190,0.8180,10291.00,60,0.0000
10225,10253,1,30,1,2,4,4,2,1,36,...,3,3,5281.00,0.00,5281.00,0.5350,0.7220,8395.00,62,0.0000


In [120]:
df['income_id'].value_counts()

income_id
1    3574
2    1804
4    1552
3    1427
6    1127
5     743
Name: count, dtype: int64

In [121]:
df['education_id'].value_counts()

education_id
4    3134
2    2028
7    1534
1    1504
3    1032
5     530
6     465
Name: count, dtype: int64

In [122]:

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 10227 entries, 0 to 10226
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   creditcard_churn_id  10227 non-null  int64  
 1   churn                10227 non-null  int64  
 2   age                  10227 non-null  int64  
 3   gender               10227 non-null  int64  
 4   dependents           10227 non-null  int64  
 5   education_id         10227 non-null  int64  
 6   marital_id           10227 non-null  int64  
 7   income_id            10227 non-null  int64  
 8   card_type_id         10227 non-null  int64  
 9   relationship_months  10227 non-null  int64  
 10  product_count        10227 non-null  int64  
 11  inactive_months      10227 non-null  int64  
 12  contact_count        10227 non-null  int64  
 13  credit_limit         10227 non-null  float64
 14  revolving_balance    10227 non-null  float64
 15  available_credit     10227 non-null  floa

In [123]:
known_df = df[df['income_id'] != 6]
unknown_df = df[df['income_id'] == 6]

print(known_df.shape)
print(unknown_df.shape)

(9100, 21)
(1127, 21)


In [124]:
X = known_df.drop(columns=['income_id'])
y = known_df['income_id']

print(X.shape)
print(y.shape)

(9100, 20)
(9100,)


In [125]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(7280, 20)
(1820, 20)
(7280,)
(1820,)


In [126]:
def print_split_report(X_train, X_test, y_train, y_test):
    # [📊 요약 출력 - 팀 공통 규격]
    print(f"\n{'Train / Test Split Summary':^50}")
    print("="*50)
    print(f"X_train shape : {X_train.shape}")
    print(f"X_test  shape : {X_test.shape}")
    print("-"*50)
    print(f"y_train shape : {y_train.shape}")
    print(f"y_test  shape : {y_test.shape}")
    print("-"*50)
    print("Target Distribution (Train)")
    print(y_train.value_counts().to_string())
    print("-"*50)
    print("Target Distribution (Test)")
    print(y_test.value_counts().to_string())
    print("-"*50)
    print("Train Ratio")
    print(y_train.value_counts(normalize=True).to_string())
    print("-"*50)
    print("Test Ratio")
    print(y_test.value_counts(normalize=True).to_string())
    print("="*50)

print_split_report(X_train, X_test, y_train, y_test)


            Train / Test Split Summary            
X_train shape : (7280, 20)
X_test  shape : (1820, 20)
--------------------------------------------------
y_train shape : (7280,)
y_test  shape : (1820,)
--------------------------------------------------
Target Distribution (Train)
income_id
1    2859
2    1443
4    1242
3    1142
5     594
--------------------------------------------------
Target Distribution (Test)
income_id
1    715
2    361
4    310
3    285
5    149
--------------------------------------------------
Train Ratio
income_id
1    0.392720
2    0.198214
4    0.170604
3    0.156868
5    0.081593
--------------------------------------------------
Test Ratio
income_id
1    0.392857
2    0.198352
4    0.170330
3    0.156593
5    0.081868


In [127]:
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_score
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.multiclass import type_of_target


def evaluate_predictions(y_true, y_pred, y_proba, classes=None, average="macro", dataset_name="Dataset"):
    target_type = type_of_target(y_true)

    result = {
        "dataset": dataset_name,
        "target_type": target_type,
        "accuracy": accuracy_score(y_true, y_pred),
    }

    # y_proba shape 보정
    y_proba = np.asarray(y_proba)

    if target_type == "binary":
        result["recall"] = recall_score(y_true, y_pred)
        result["f1"] = f1_score(y_true, y_pred)

        # (n_samples, 2) 이면 positive class 확률만 사용
        if y_proba.ndim == 2 and y_proba.shape[1] == 2:
            y_score = y_proba[:, 1]
        # 이미 (n_samples,) 형태면 그대로 사용
        elif y_proba.ndim == 1:
            y_score = y_proba
        else:
            print(f"Binary Classification Shape Error: y_proba.shape = {y_proba.shape}")
            return {}

        result["roc_auc"] = roc_auc_score(y_true, y_score)
        result["pr_auc"] = average_precision_score(y_true, y_score)
        result["precision_score"] = precision_score(y_true, y_pred)

    elif target_type == "multiclass":
        result["recall"] = recall_score(y_true, y_pred, average=average)
        result["f1"] = f1_score(y_true, y_pred, average=average)

        if y_proba.ndim != 2:
            raise ValueError(f"Multiclass classification에서는 y_proba가 2차원이어야 합니다: {y_proba.shape}")

        result["roc_auc"] = roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average=average
        )

        if classes is None:
            classes = np.unique(y_true)

        y_true_bin = label_binarize(y_true, classes=classes)
        result["pr_auc"] = average_precision_score(
            y_true_bin,
            y_proba,
            average=average
        )
        
        result["precision_score"] = precision_score(y_true, y_pred, average=average)

    else:
        raise ValueError(f"지원하지 않는 target type입니다: {target_type}")

    return result

def print_report(res):
    print("=" * 20, f"{res["dataset"]}", "="*20)
    print(f"{"target type":>20}: {res["target_type"]}")
    for k in [key for key in res if key not in ["dataset", "target_type"]]:
        print(f"{k:>20}: {res[k]:.6f}")

In [128]:
from xgboost import XGBClassifier

xgb_income = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist"
)

xgb_income.fit(X_train, y_train - 1)

y_train_pred_enc = xgb_income.predict(X_train)          # 0~4
y_train_proba = xgb_income.predict_proba(X_train)       # 각 클래스 확률
y_train_pred = y_train_pred_enc + 1                          # 1~5로 복원

y_test_pred_enc = xgb_income.predict(X_test)          # 0~4
y_test_proba = xgb_income.predict_proba(X_test)       # 각 클래스 확률
y_test_pred = y_test_pred_enc + 1     

In [129]:
income_train_result = evaluate_predictions(
    y_true=y_train,
    y_pred=y_train_pred,
    y_proba=y_train_proba,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Train XGBoost"
)

income_test_result = evaluate_predictions(
    y_true=y_test,
    y_pred=y_test_pred,
    y_proba=y_test_proba,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Test XGBoost"
)

print_report(income_train_result)
print_report(income_test_result)

==================== Income Train XGBoost ====================
         target type: multiclass
            accuracy: 0.807143
              recall: 0.760317
                  f1: 0.770167
             roc_auc: 0.975812
              pr_auc: 0.904693
     precision_score: 0.827232
==================== Income Test XGBoost ====================
         target type: multiclass
            accuracy: 0.585714
              recall: 0.464081
                  f1: 0.454376
             roc_auc: 0.839420
              pr_auc: 0.498063
     precision_score: 0.491800


In [ ]:
from xgboost import XGBClassifier

def my_grid_search(n_estim, min_child_len, max_depth):
    max_score = 0
    best_params = []

    for i in n_estim:
        for j in min_child_len:
            for k in max_depth:
                print(f"n_estimators: {i}, min_child_weight: {j}, max_depth: {k}")
                xgb_income = XGBClassifier(
                    objective="multi:softprob",
                    num_class=5,
                    eval_metric="mlogloss",
                    n_estimators=i,
                    max_depth=k,
                    learning_rate=0.05,
                    min_child_weight=j,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    random_state=42,
                    tree_method="hist"
                )
                xgb_income.fit(X_train, y_train - 1)
                
                y_test_pred_enc = xgb_income.predict(X_test)
                y_test_pred = y_test_pred_enc + 1    
                
                sc = accuracy_score(y_test, y_test_pred)
                if max_score < sc:
                    max_score = sc
                    best_params = [i, j, k]

    print(f"Best Score: {max_score:.6f} with params: n_estimators={best_params[0]}, min_child_weight={best_params[1]}, max_depth={best_params[2]}")

my_grid_search([50], [3, 4, 5], [8, 9, 10, 11])

n_estimators: 50, min_child_weight: 3, max_depth: 8
n_estimators: 50, min_child_weight: 3, max_depth: 9
n_estimators: 50, min_child_weight: 3, max_depth: 10
n_estimators: 50, min_child_weight: 3, max_depth: 11
n_estimators: 50, min_child_weight: 4, max_depth: 8
n_estimators: 50, min_child_weight: 4, max_depth: 9
n_estimators: 50, min_child_weight: 4, max_depth: 10
n_estimators: 50, min_child_weight: 4, max_depth: 11
n_estimators: 50, min_child_weight: 5, max_depth: 8
n_estimators: 50, min_child_weight: 5, max_depth: 9
n_estimators: 50, min_child_weight: 5, max_depth: 10
n_estimators: 50, min_child_weight: 5, max_depth: 11
Best Score: 0.598901 with params: n_estimators=50, min_child_weight=3, max_depth=9


In [ ]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

params = {
    'n_estimators': [100, 200, 300],
    'min_child_weight': [1, 3, 5],
    'max_depth': [3, 5, 7]
}

xgb_income = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="aucpr",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist"
)

grid_search = GridSearchCV(
    estimator=xgb_income,
    param_grid=params,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train - 1)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 3 folds for each of 27 candidates, totalling 81 fits


[CV] END ..max_depth=3, min_child_weight=3, n_estimators=100; total time=   0.7s
[CV] END ..max_depth=3, min_child_weight=1, n_estimators=100; total time=   0.7s
[CV] END ..max_depth=3, min_child_weight=1, n_estimators=100; total time=   0.7s
[CV] END ..max_depth=3, min_child_weight=3, n_estimators=100; total time=   0.7s
[CV] END ..max_depth=3, min_child_weight=1, n_estimators=100; total time=   0.9s
[CV] END ..max_depth=3, min_child_weight=3, n_estimators=100; total time=   0.9s
[CV] END ..max_depth=3, min_child_weight=1, n_estimators=200; total time=   1.2s
[CV] END ..max_depth=3, min_child_weight=5, n_estimators=100; total time=   0.6s
[CV] END ..max_depth=3, min_child_weight=5, n_estimators=100; total time=   0.6s
[CV] END ..max_depth=3, min_child_weight=3, n_estimators=200; total time=   1.2s
[CV] END ..max_depth=3, min_child_weight=1, n_estimators=200; total time=   1.2s
[CV] END ..max_depth=3, min_child_weight=3, n_estimators=200; total time=   1.2s
[CV] END ..max_depth=3, min_

In [ ]:
params = {
    'n_estimators': [50, 100, 150],
    'min_child_weight': [4, 5, 6],
}

xgb_income = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    eval_metric="mlogloss",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    max_depth=5,
    random_state=42,
    tree_method="hist"
)

grid_search = GridSearchCV(
    estimator=xgb_income,
    param_grid=params,
    scoring="accuracy",
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X_train, y_train - 1)
print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END ................min_child_weight=4, n_estimators=50; total time=   0.5s
[CV] END ................min_child_weight=5, n_estimators=50; total time=   0.4s
[CV] END ................min_child_weight=5, n_estimators=50; total time=   0.5s
[CV] END ................min_child_weight=4, n_estimators=50; total time=   0.6s
[CV] END ...............min_child_weight=5, n_estimators=100; total time=   0.9s
[CV] END ................min_child_weight=4, n_estimators=50; total time=   0.8s
[CV] END ................min_child_weight=5, n_estimators=50; total time=   0.8s
[CV] END ...............min_child_weight=4, n_estimators=150; total time=   1.3s
[CV] END ...............min_child_weight=5, n_estimators=100; total time=   1.3s
[CV] END ................min_child_weight=6, n_estimators=50; total time=   0.6s
[CV] END ...............min_child_weight=4, n_estimators=100; total time=   0.9s
[CV] END ...............min_child_weight=4, n_est

In [130]:
xgb_income = XGBClassifier(
    objective="multi:softprob",
    n_estimators=50,
    min_child_weight=3,
    num_class=5,
    eval_metric="mlogloss",
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    max_depth=9,
    random_state=42,
    tree_method="hist"
)

xgb_income.fit(X_train, y_train - 1)

y_train_pred_enc = xgb_income.predict(X_train)          # 0~4
y_train_proba = xgb_income.predict_proba(X_train)       # 각 클래스 확률
y_train_pred = y_train_pred_enc + 1                          # 1~5로 복원

y_test_pred_enc = xgb_income.predict(X_test)          # 0~4
y_test_proba = xgb_income.predict_proba(X_test)       # 각 클래스 확률
y_test_pred = y_test_pred_enc + 1    

In [131]:
income_train_result = evaluate_predictions(
    y_true=y_train,
    y_pred=y_train_pred,
    y_proba=y_train_proba,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Train XGBoost"
)

income_test_result = evaluate_predictions(
    y_true=y_test,
    y_pred=y_test_pred,
    y_proba=y_test_proba,
    classes=[1, 2, 3, 4, 5],
    average="macro",
    dataset_name="Income Test XGBoost"
)

print_report(income_train_result)
print_report(income_test_result)

==================== Income Train XGBoost ====================
         target type: multiclass
            accuracy: 0.804396
              recall: 0.748059
                  f1: 0.761530
             roc_auc: 0.976484
              pr_auc: 0.910759
     precision_score: 0.840402
==================== Income Test XGBoost ====================
         target type: multiclass
            accuracy: 0.598901
              recall: 0.470384
                  f1: 0.460671
             roc_auc: 0.842808
              pr_auc: 0.511087
     precision_score: 0.520618


In [145]:
y_train_data = (~y_train.isin([1, 2])).astype(int)
y_test_data = (~y_test.isin([1, 2])).astype(int)

print(np.unique(y_train_data))
print(np.unique(y_test_data))
print(y_train_data.value_counts())
print(y_test_data.value_counts())

[0 1]
[0 1]
income_id
0    4302
1    2978
Name: count, dtype: int64
income_id
0    1076
1     744
Name: count, dtype: int64


In [ ]:
xgb_income_bin = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=42,
    tree_method="hist"
)

xgb_income_bin.fit(X_train, y_train_data)

In [147]:
y_train_pred = xgb_income_bin.predict(X_train)
y_train_proba = xgb_income_bin.predict_proba(X_train)

y_test_pred = xgb_income_bin.predict(X_test)
y_test_proba = xgb_income_bin.predict_proba(X_test)

print("y_train_pred shape:", y_train_pred.shape)
print("y_train_proba shape:", y_train_proba.shape)
print("unique y_train_pred:", np.unique(y_train_pred))

y_train_pred shape: (7280,)
y_train_proba shape: (7280, 2)
unique y_train_pred: [0 1]


In [148]:
income_train_result = evaluate_predictions(
    y_true=y_train_data,
    y_pred=y_train_pred,
    y_proba=y_train_proba,
    classes=[0, 1],
    average="macro",
    dataset_name="Income Train XGBoost"
)

income_test_result = evaluate_predictions(
    y_true=y_test_data,
    y_pred=y_test_pred,
    y_proba=y_test_proba,
    classes=[0, 1],
    average="macro",
    dataset_name="Income Test XGBoost"
)

print_report(income_train_result)
print_report(income_test_result)

==================== Income Train XGBoost ====================
         target type: binary
            accuracy: 0.951374
              recall: 0.990262
                  f1: 0.943378
             roc_auc: 0.995500
              pr_auc: 0.993302
     precision_score: 0.900733
==================== Income Test XGBoost ====================
         target type: binary
            accuracy: 0.902747
              recall: 0.948925
                  f1: 0.888609
             roc_auc: 0.967565
              pr_auc: 0.949256
     precision_score: 0.835503
